In [1]:
import os
os.chdir("../")
%pwd

'/home/minh_khai/skin_disease/skin-disease'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TransformationMetrics:
    image_size:  int
    train_ratio: float
    valid_ratio: float
    
@dataclass(frozen=True)
class TransformationConfig:
    root_dir:       Path
    data_dirs:      tuple
    out_train_dir:  Path
    out_valid_dir:  Path
    out_infer_dir:  Path
    metrics:        TransformationMetrics

In [3]:
import os
from itertools import chain
from pathlib import Path

# --- utils/helpers ----
def _parse_env_sources(env_key: str) -> list[dict]:
    """Parse comma-separated SOURCES env var into list of dicts."""
    raw = os.getenv(env_key, "")
    if not raw.strip():
        return []
    
    sources = []
    for entry in raw.split(","):
        parts = entry.strip().split(":")
        if len(parts) != 3:
            continue
        
        name, src_type, source = parts
        sources.append({"name": name, "src_type": src_type, "source": source})
    return sources

def _build_data_dirs(ingestion_root: Path, *source_lists: list[dict]) -> list[dict]:
    """Build data_dirs from multiple source lists."""
    return [
        {"path": ingestion_root / s["name"], "name": s["name"]}
        for s in chain(*source_lists)
    ]

In [4]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from core.constants import *
from core import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_transformation_config(self) -> TransformationConfig:
        config = self.config.transformation_config
        params = self.params.transformation_params
        create_directories([config.root_dir])

        data_dirs = _build_data_dirs(
            Path(self.config.ingestion_config.root_dir),
            _parse_env_sources("LOCAL_SOURCES"),
            _parse_env_sources("KAGGLE_SOURCES"),
        )

        return TransformationConfig(
            root_dir        = Path(config.root_dir),
            data_dirs       = data_dirs,
            out_train_dir   = Path(config.root_dir) / "train",
            out_valid_dir   = Path(config.root_dir) / "valid",
            out_infer_dir   = Path(config.root_dir) / "infer",
            metrics = TransformationMetrics(
                image_size  = params.image_size,
                train_ratio = params.train_ratio,
                valid_ratio = params.valid_ratio
            ),
        )

In [5]:
from abc import ABC, abstractmethod
import pandas as pd

class BaseTransformer(ABC):
    @abstractmethod
    def transform(self) -> pd.DataFrame:
        """Transform source data → standard schema DataFrame"""
        pass

In [22]:
from pathlib import Path
import pandas as pd
from core.logger import logger

class KaggleTransformer(BaseTransformer):
    def __init__(self, src_path: str):
        self.src_path = src_path
    
    def transform(self) -> pd.DataFrame:
        actual_path = self.src_path
        while True:
            subdirs = [d for d in actual_path.iterdir() if d.is_dir()]
            if len(subdirs) == 1:
                actual_path = subdirs[0]  # going through folders until reaching data
            else:
                break
        
        records = []
        for class_dir in actual_path.iterdir():
            if not class_dir.is_dir():
                continue
        
            label = class_dir.name
            for img_path in class_dir.glob("*.*"):
                records.append({"image_path": str(img_path), "label": label})

        df = pd.DataFrame(records)
        logger.info(f"KaggleTransformer: loaded {len(df)} records from {actual_path}")
        return df

In [23]:
import pandas as pd

class TransformerFactory:
    _transformers = {
        "SKIN_DISEASES": KaggleTransformer,
    }
    
    @staticmethod
    def create(name: str, path: str) -> pd.DataFrame:
        cls = TransformerFactory._transformers.get(name)
        if not cls:
            raise ValueError(f"No transformer registered for: '{name}'. Add it to TransformerFactory._transformers.")
        return cls(src_path=path).transform()

In [26]:
import os, sys, numpy as np
from PIL import Image
import albumentations as A

from core.logger import logger
from core.exception import CustomException

class Transformation:
    def __init__(self, config: TransformationConfig):
        self.config   = config
        self.dfs = [
            (TransformerFactory.create(d["name"], d["path"]), d)
            for d in self.config.data_dirs
            if d["name"] in TransformerFactory._transformers
        ]
        logger.info(f"Loaded {len(self.dfs)} transformer(s): {[d['name'] for _, d in self.dfs]}")
    
    def _prepare_output_dirs(self):
        for d in [self.config.out_train_dir, self.config.out_valid_dir, self.config.out_infer_dir]:
            os.makedirs(d, exist_ok=True)
            
    def _get_resizer(self):
        size = self.config.metrics.image_size
        return A.Compose([A.Resize(size, size)])
    
    def _save_split(self, df_split, out_dir, resizer):
        for _, row in df_split.iterrows():
            label_dir = out_dir / row["label"]
            os.makedirs(label_dir, exist_ok=True)
            
            img = Image.open(row["image_path"]).convert("RGB")
            resized = resizer(image=np.array(img))["image"]

            out_path = label_dir / Path(row["image_path"]).name
            Image.fromarray(resized).save(out_path)
    
    def transform(self):
        try:
            self._prepare_output_dirs()
            metrics = self.config.metrics
            resizer = self._get_resizer()
            
            for df, d in self.dfs:
                name = d["name"].lower()
                df   = df.sample(frac=1, random_state=42).reset_index(drop=True)
                n    = len(df)
                
                train_end = int(n * metrics.train_ratio)
                valid_end = int(n * (metrics.train_ratio + metrics.valid_ratio))
                
                train_df = df.iloc[:train_end]
                valid_df = df.iloc[train_end:valid_end]
                infer_df = df.iloc[valid_end:]
                
                logger.info(f"{name}: train={len(train_df)} | valid={len(valid_df)} | infer={len(infer_df)}")
                
                self._save_split(train_df, self.config.out_train_dir, resizer)
                self._save_split(valid_df, self.config.out_valid_dir, resizer)
                self._save_split(infer_df, self.config.out_infer_dir, resizer)
            
        except Exception as e:
            raise CustomException(e, sys)

In [27]:
try:
    cfg_manager = ConfigurationManager()
    transform   = cfg_manager.get_transformation_config()
    transform   = Transformation(config=transform)
    transform.transform()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-16 10:32:24,604: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-16 10:32:24,607: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-16 10:32:24,609: INFO: __init__: created directory at: artifacts]
[2026-06-16 10:32:24,610: INFO: __init__: created directory at: artifacts/transformation]
[2026-06-16 10:32:24,750: INFO: 1645623499: KaggleTransformer: loaded 27153 records from artifacts/ingestion/SKIN_DISEASES/IMG_CLASSES]
[2026-06-16 10:32:24,755: INFO: 756844384: Loaded 1 transformer(s): ['SKIN_DISEASES']]
[2026-06-16 10:32:24,762: INFO: 756844384: skin_diseases: train=19007 | valid=4073 | infer=4073]
